# How many of the 377 raw features invert?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [1]:
from pathlib import Path

import polars as pl

from fraud_detection.evaluation.time_consistency import scan, time_windows

C = [f"C{i}" for i in range(1, 15)]
D = [f"D{i}" for i in range(1, 16)]
M = [f"M{i}" for i in range(1, 10)]
V = [f"V{i}" for i in range(1, 340)]
FEATURES = C + D + M + V

# ../input on a Kaggle kernel, kaggle/raw locally (see kaggle/download.py)
CANDIDATES = [
    Path("../input/ieee-fraud-detection/train_transaction.csv"),
    Path("../../kaggle/raw/train_transaction.csv"),
    Path("../kaggle/raw/train_transaction.csv"),
    Path("kaggle/raw/train_transaction.csv"),
]
csv = next((p for p in CANDIDATES if p.exists()), None)
if csv is None:
    raise FileNotFoundError(
        "train_transaction.csv not found. On Kaggle, add the ieee-fraud-detection "
        "competition data to this notebook; locally, run `uv run python kaggle/download.py`."
    )

df = pl.read_csv(
    csv,
    columns=["TransactionDT", "isFraud"] + FEATURES,
    schema_overrides={c: pl.Float32 for c in V},
)
print(f"{len(df):,} rows, {len(FEATURES)} features")

590,540 rows, 377 features


In [2]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DAY = 86400

# Categorical slots
PASS, INVERTED, WEAK = "#2a78d6", "#eb6834", "#1baf7a"
VERDICT_COLOR = {"pass": PASS, "inverted": INVERTED, "weak": WEAK}
BLUE_LIGHT = "#9ec5f4"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"


In [3]:
train, holdout = time_windows(df, "TransactionDT", train=(0.0, 0.17), holdout=(0.83, 1.0))
print(f"train {len(train):,}   skipped {len(df) - len(train) - len(holdout):,}   holdout {len(holdout):,}")

train 100,392   skipped 389,755   holdout 100,393


In [4]:

by_day = (
    df.with_columns((pl.col("TransactionDT") // DAY).cast(pl.Int64).alias("day"))
    .group_by("day")
    .agg(pl.len().alias("volume"), pl.col("isFraud").mean().alias("rate"))
    .sort("day")
)
day = by_day["day"].to_numpy()
volume = by_day["volume"].to_numpy()
rate = by_day["rate"].to_numpy()
t_hi = train["TransactionDT"].max() / DAY
h_lo = holdout["TransactionDT"].min() / DAY

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Two windows, and the days skipped between them", ""))

fig.add_trace(go.Scatter(x=day, y=volume, mode='lines', name='volume', line={"color": MUTED, "width": 1}), row=1, col=1)
fig.add_trace(go.Scatter(x=day, y=rate, mode='lines', name='rate', line={"color": MUTED, "width": 1}), row=2, col=1)

fig.add_vrect(x0=day.min(), x1=t_hi, fillcolor=PASS, opacity=0.13, line_width=0, row="all", col=1)
fig.add_vrect(x0=h_lo, x1=day.max(), fillcolor=PASS, opacity=0.13, line_width=0, row="all", col=1)

fig.add_annotation(x=t_hi/2, y=0.9, xref="x1", yref="y domain", text="train", showarrow=False, font={"color": PASS}, row=1, col=1)
fig.add_annotation(x=(t_hi+h_lo)/2, y=0.9, xref="x1", yref="y domain", text="skipped", showarrow=False, font={"color": MUTED}, row=1, col=1)
fig.add_annotation(x=(h_lo+day.max())/2, y=0.9, xref="x1", yref="y domain", text="holdout", showarrow=False, font={"color": PASS}, row=1, col=1)

fig.update_yaxes(title_text="transactions / day", row=1, col=1)
fig.update_yaxes(title_text="fraud rate", row=2, col=1)
fig.update_xaxes(title_text="day of the period", row=2, col=1)

fig.update_layout(height=450, width=750, showlegend=False, plot_bgcolor=SURFACE, paper_bgcolor=SURFACE)
fig.show()


In [5]:
scan(train, holdout, ["C3", "C7"], "isFraud").select(
    ["feature", "verdict", "auc_train", "auc_holdout", "delta"]
)

feature,verdict,auc_train,auc_holdout,delta
str,str,f64,f64,f64
"""C3""","""weak""",0.5055,0.502,-0.0035
"""C7""","""pass""",0.6499,0.6638,0.0139


In [6]:
block = scan(train, holdout, [f"V{i}" for i in range(322, 340)], "isFraud")
print(block["verdict"].value_counts().sort("count", descending=True), "\n")
block.select(["feature", "verdict", "auc_train", "auc_holdout", "delta"])

shape: (2, 2)
┌──────────┬───────┐
│ verdict  ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ inverted ┆ 12    │
│ pass     ┆ 6     │
└──────────┴───────┘ 



feature,verdict,auc_train,auc_holdout,delta
str,str,f64,f64,f64
"""V334""","""inverted""",0.5672,0.4644,-0.1028
"""V335""","""inverted""",0.5709,0.4691,-0.1017
"""V336""","""inverted""",0.5698,0.4709,-0.0989
"""V325""","""inverted""",0.5613,0.464,-0.0972
"""V337""","""inverted""",0.5675,0.473,-0.0946
…,…,…,…,…
"""V333""","""pass""",0.5834,0.493,-0.0905
"""V332""","""pass""",0.5833,0.4974,-0.0859
"""V322""","""pass""",0.5669,0.49,-0.0769


In [7]:
import plotly.graph_objects as go

b = block.sort("auc_holdout")
features = b["feature"].to_list()
auc_train = b["auc_train"].to_list()
auc_holdout = b["auc_holdout"].to_list()

fig = go.Figure()

for i, (f, t, h) in enumerate(zip(features, auc_train, auc_holdout)):
    fig.add_trace(go.Scatter(x=[t, h], y=[f, f], mode='lines', line={"color": GRID, "width": 2}, showlegend=False))

fig.add_trace(go.Scatter(x=auc_train, y=features, mode='markers', marker={"size": 8, "color": BLUE_LIGHT, "line": {"width": 1.5, "color": SURFACE}}, name='train window'))
fig.add_trace(go.Scatter(x=auc_holdout, y=features, mode='markers', marker={"size": 8, "color": PASS, "line": {"width": 1.5, "color": SURFACE}}, name='holdout window'))

fig.add_vline(x=0.5, line_width=1.5, line_color=MUTED)
fig.add_annotation(x=0.5, y=len(b)-1, text="no signal", showarrow=False, xanchor="left", font={"color": MUTED, "size": 10})

fig.update_layout(title="V322-V339: all eighteen move the same way, across the 0.5 line",
                  xaxis_title="single-feature AUC",
                  height=500, width=750,
                  legend={"yanchor": "middle", "y": 0.5, "xanchor": "center", "x": 0.5},
                  plot_bgcolor=SURFACE, paper_bgcolor=SURFACE)
fig.show()


## 3. Full Sweep (377 Features)

Fit LightGBM for each of the 377 features (500 trees, 8 leaves).

In [8]:
report = scan(train, holdout, FEATURES, "isFraud", n_jobs=-1)
report.write_csv("time_consistency_report.csv")
report["verdict"].value_counts().sort("count", descending=True)

verdict,count
str,u32
"""pass""",327
"""inverted""",30
"""weak""",18
"""degenerate""",2


In [9]:
inverted = report.filter(pl.col("verdict") == "inverted")
non_v = [f for f in inverted["feature"].to_list() if not f.startswith("V")]
print("non-V columns inverted:", non_v or "none")
inverted.select(["feature", "auc_train", "auc_holdout", "delta", "null_rate_train"]).head(15)

non-V columns inverted: none


feature,auc_train,auc_holdout,delta,null_rate_train
str,f64,f64,f64,f64
"""V161""",0.5789,0.4656,-0.1133,0.6833
"""V163""",0.5787,0.4674,-0.1113,0.6833
"""V159""",0.5734,0.4627,-0.1107,0.6833
"""V160""",0.5862,0.4783,-0.1079,0.6833
"""V162""",0.5782,0.4704,-0.1078,0.6833
…,…,…,…,…
"""V138""",0.5687,0.4703,-0.0984,0.6833
"""V325""",0.5613,0.464,-0.0972,0.682
"""V337""",0.5675,0.473,-0.0946,0.682


In [10]:
import plotly.graph_objects as go

ev = report.drop_nulls(subset=["auc_train", "auc_holdout"])
lo = min(ev["auc_train"].min(), ev["auc_holdout"].min()) - 0.012
hi = max(ev["auc_train"].max(), ev["auc_holdout"].max()) + 0.012

fig = go.Figure()
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode='lines', line={"color": MUTED, "width": 1}, showlegend=False))

for verdict in ("pass", "weak", "inverted"):
    s = ev.filter(pl.col("verdict") == verdict)
    fig.add_trace(go.Scatter(x=s["auc_train"], y=s["auc_holdout"], mode='markers',
                             marker={"size": 6, "color": VERDICT_COLOR[verdict], "line": {"width": 0.7, "color": SURFACE}},
                             name=f"{verdict} ({len(s)})"))

for r in ev.sort("delta").head(3).iter_rows(named=True):
    fig.add_annotation(x=r["auc_train"], y=r["auc_holdout"], text=r["feature"],
                       showarrow=False, xanchor="left", yanchor="top", xshift=5, yshift=-5, font={"color": INK, "size": 10})

fig.add_hline(y=0.5, line_width=1, line_color=GRID)
fig.add_vline(x=0.5, line_width=1, line_color=GRID)

fig.add_annotation(x=hi - 0.004, y=0.502, text="inversions sit below this line",
                   showarrow=False, xanchor="right", yanchor="bottom", font={"color": MUTED, "size": 10})

fig.update_layout(title="Every feature, both windows",
                  xaxis_title="AUC, train window",
                  yaxis_title="AUC, holdout window",
                  xaxis={"range": [lo, hi]},
                  yaxis={"range": [lo, hi], "scaleanchor": "x", "scaleratio": 1},
                  height=620, width=640,
                  legend={"yanchor": "top", "y": 1, "xanchor": "left", "x": 0},
                  plot_bgcolor=SURFACE, paper_bgcolor=SURFACE)
fig.show()


V310 has the worst delta (-0.1218). 22 columns degrade significantly.